# Learning Objectives

In this notebook, you will craft sophisticated ETL jobs that interface with a variety of common data sources, such as 
- REST APIs (HTTP endpoints)
- RDBMS
- Hive tables (managed tables)
- Various file formats (csv, json, parquet, etc.)


# Interview Questions

As you progress through the practice, attempt to answer the following questions:

## Columnar File
- What is a columnar file format and what advantages does it offer?
- Why is Parquet frequently used with Spark and how does it function?
- How do you read/write data from/to a Parquet file using a DataFrame?

## Partitions
- How do you save data to a file system by partitions? (Hint: Provide the code)
- How and why can partitions reduce query execution time? (Hint: Give an example)

## JDBC and RDBMS
- How do you load data from an RDBMS into Spark? (Hint: Discuss the steps and JDBC)

## REST API and HTTP Requests
- How can Spark be used to fetch data from a REST API? (Hint: Discuss making API requests)

## ETL Job One: Parquet file
### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Data transformation requirements https://pgexercises.com/questions/aggregates/fachoursbymonth.html

### Load
Load data into a parquet file

### What is Parquet? 

Columnar files are an important technique for optimizing Spark queries. Additionally, they are often tested in interviews.
- https://www.youtube.com/watch?v=KLFadWdomyI
- https://www.databricks.com/glossary/what-is-parquet

In [0]:
# Write your solution here
# Extract
bookings_df = spark.sql("SELECT * FROM bookings")

# Transform
result_df = (
    bookings_df
    .filter("starttime >= '2012-09-01' AND starttime < '2012-10-01'")
    .groupBy("facid")
    .sum("slots")
    .withColumnRenamed("sum(slots)", "total_slots")
    .orderBy("total_slots")
)

display(result_df)

# Load
result_df.write.mode("overwrite").parquet("/Volumes/workspace/default/pgexercise_volume/fachoursbymonth.parquet")


facid,total_slots
5,122
3,422
7,426
8,471
6,540
2,570
1,588
0,591
4,648


## ETL Job Two: Partitions

### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Transform the data https://pgexercises.com/questions/joins/threejoin.html

### Load
Partition the result data by facility column and then save to `threejoin_delta` managed table. Additionally, they are often tested in interviews.

hint: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.partitionBy.html

What are paritions? 

Partitions are an important technique to optimize Spark queries
- https://www.youtube.com/watch?v=hvF7tY2-L3U&t=268s

In [0]:
# Write your solution here
# Extract
bookings_df = spark.sql("SELECT * FROM bookings")
members_df = spark.sql("SELECT * FROM members")
facilities_df = spark.sql("SELECT * FROM facilities")

# Transform
from pyspark.sql.functions import concat, lit

result_df = (
    bookings_df.alias("b")
    .join(members_df.alias("m"), "memid")
    .join(facilities_df.alias("f"), "facid")
    .filter("f.name LIKE 'Tennis Court%' AND m.memid != 0")
    .select(
        concat("m.firstname", lit(" "), "m.surname").alias("member"),
        "f.name",
        "f.facid"
    )
    .distinct()
)

display(result_df)

# Load - partitioned by facility
spark.sql("DROP TABLE IF EXISTS threejoin_delta")
(
    result_df.write
    .mode("overwrite")
    .partitionBy("facid")
    .format("delta")
    .saveAsTable("threejoin_delta")
)


member,name,facid
Anne Baker,Tennis Court 2,1
Ramnaresh Sarwin,Tennis Court 1,0
Gerald Butters,Tennis Court 2,1
Burton Tracy,Tennis Court 1,0
Nancy Dare,Tennis Court 1,0
Jemima Farrell,Tennis Court 2,1
Charles Owen,Tennis Court 2,1
Timothy Baker,Tennis Court 2,1
Ramnaresh Sarwin,Tennis Court 2,1
Douglas Jones,Tennis Court 1,0


## ETL Job Three: HTTP Requests

### Extract
Extract daily stock price data price from the following companies, Google, Apple, Microsoft, and Tesla. 

Data Source
- API: https://rapidapi.com/alphavantage/api/alpha-vantage
- Endpoint: GET `TIME_SERIES_DAILY`

Sample HTTP request

```
curl --request GET \
	--url 'https://alpha-vantage.p.rapidapi.com/query?function=TIME_SERIES_DAILY&symbol=TSLA&outputsize=compact&datatype=json' \
	--header 'X-RapidAPI-Host: alpha-vantage.p.rapidapi.com' \
	--header 'X-RapidAPI-Key: [YOUR_KEY]'

```

Sample Python HTTP request

```
import requests

url = "https://alpha-vantage.p.rapidapi.com/query"

querystring = {
    "function":"TIME_SERIES_DAILY",
    "symbol":"IBM",
    "datatype":"json",
    "outputsize":"compact"
}

headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": "[YOUR_KEY]"
}

response = requests.get(url, headers=headers, params=querystring)

data = response.json()

# Now 'data' contains the daily time series data for "IBM"
```

### Transform
Find **weekly** max closing price for each company.

hints: 
  - Use a `for-loop` to get stock data for each company
  - Use the spark `union` operation to concat all data into one DF
  - create a new `week` column from the data column
  - use `group by` to calcualte max closing price

### Load
- Partition `DF` by company
- Load the DF in to a managed table called, `max_closing_price_weekly`

In [0]:
# Write your solution here

import time
import requests
from pyspark.sql import Row
from pyspark.sql.functions import to_date, date_trunc, col, max as spark_max

API_KEY = "d0c9dca170msh7fb73707e184827p1546a3jsn6b9336f510ee"
companies = ["GOOGL", "AAPL", "MSFT", "TSLA"]

url = "https://alpha-vantage.p.rapidapi.com/query"
headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": API_KEY
}

all_dfs = []

for symbol in companies:
    querystring = {
        "function": "TIME_SERIES_DAILY",
        "symbol": symbol,
        "datatype": "json",
        "outputsize": "compact"
    }
    response = requests.get(url, headers=headers, params=querystring)
    data = response.json()

    time_series = data.get("Time Series (Daily)", {})
    if not time_series:
        print(f"WARNING: no data for {symbol}: {data}")
        time.sleep(15)
        continue

    rows = [
        Row(company=symbol, date=date_str, close=float(values["4. close"]))
        for date_str, values in time_series.items()
    ]
    df = spark.createDataFrame(rows)
    all_dfs.append(df)

    time.sleep(15)  # stay under the per-minute rate limit

# Union all company DataFrames
combined_df = all_dfs[0]
for df in all_dfs[1:]:
    combined_df = combined_df.union(df)

# Transform: weekly max closing price
combined_df = combined_df.withColumn("date", to_date("date"))
combined_df = combined_df.withColumn("week", date_trunc("week", col("date")))

weekly_max_df = (
    combined_df.groupBy("company", "week")
    .agg(spark_max("close").alias("max_closing_price"))
    .orderBy("company", "week")
)

display(weekly_max_df)

# Load - partitioned by company
spark.sql("DROP TABLE IF EXISTS max_closing_price_weekly")
(
    weekly_max_df.write
    .mode("overwrite")
    .partitionBy("company")
    .format("delta")
    .saveAsTable("max_closing_price_weekly")
)

company,week,max_closing_price
AAPL,2026-02-09T00:00:00.000Z,275.5
AAPL,2026-02-16T00:00:00.000Z,264.58
AAPL,2026-02-23T00:00:00.000Z,274.23
AAPL,2026-03-02T00:00:00.000Z,264.72
AAPL,2026-03-09T00:00:00.000Z,260.83
AAPL,2026-03-16T00:00:00.000Z,254.23
AAPL,2026-03-23T00:00:00.000Z,252.89
AAPL,2026-03-30T00:00:00.000Z,255.92
AAPL,2026-04-06T00:00:00.000Z,260.49
AAPL,2026-04-13T00:00:00.000Z,270.23


## ETL Job Four: RDBMS


### Extract
Extract RNA data from a public PostgreSQL database.

- https://rnacentral.org/help/public-database
- Extract 100 RNA records from the `rna` table (hint: use `limit` in your sql)
- hint: use `spark.read.jdbc` https://docs.databricks.com/external-data/jdbc.html

### Transform
We want to load the data as it so there is no transformation required.


### Load
Load the DF in to a managed table called, `rna_100_records`

In [0]:
# Write your solution here
jdbc_url = "jdbc:postgresql://hh-pgsql-public.ebi.ac.uk:5432/pfmegrnargs"

connection_properties = {
    "user": "reader",
    "password": "NWDMCE5xdipIjRrp",
    "driver": "org.postgresql.Driver"
}

# Extract - limit 100 records
query = "(SELECT * FROM rna LIMIT 100) AS rna_subset"

rna_df = spark.read.jdbc(
    url=jdbc_url,
    table=query,
    properties=connection_properties
)

display(rna_df)

# No transformation needed - load as is
spark.sql("DROP TABLE IF EXISTS rna_100_records")
rna_df.write.mode("overwrite").saveAsTable("rna_100_records")


id,upi,timestamp,userstamp,crc64,len,seq_short,seq_long,md5
22926896,URS00015DD630,2019-12-02T13:24:47.744Z,rnacen,4F0D4F3B5B6B55BA,1417,GCGGCGTGCTTAACACATGCAAGTCGAACGATGAAGCCCAGCTTGCTGGGTGGATTAGTGGCGAACGGGTGAGTAACACGTGAGTAACCTGCCCCCGACTTTGGGATAAGCCCGGGAAACTGGGTCTAATACCGGATATGACTTTCCACCGCATGGTGGGTTGTTGAAAGATTTATCGGTGGGGGATGGACTCGCGGCCTATCAGCTTGTTGGTGAGGTAATGGCTCACCAAGGCGACGACGGGTAGCCGGCCTGAGAGGGTGACCGGCCACACTGGGACTGAGACACGGCCCAGACTCCTACGGGAGGCAGCAGTGGGGAATATTGCACAATGGGCGGAAGCCTGATGCAGCGACGCCGCGTGAGGGATGACGGCCTTCGGGTTGTAAACCTCTTTCAGTAGGGAAGAAGCGAAAGTGACGGTACCTGCAGAAGAAGCGCCGGCTAACTACGTGCCAGCAGCCGCGGTAATACGTAGGGCGCAAGCGTTATCCGGATTTATTGGGCGTAAAGAGCTCGTAGGCGGTTTGTCGCGTCTGCCGTGAAAGTCCGAGGCTCAACCTCGGATCTGCGGTGGGTACGGGCAGACTAGAGTGATGTAGGGGAGACTGGAATTCCTGGTGTAGCGGTGAAATGCGCAGATATCAGGAGGAACACCGATGGCGAAGGCAGGTCTCTGGGCATTTACTGACGCTGAGGAGCGAAAGCATGGGGAGCGAACAGGATTAGATACCCTGGTAGTCCATGCCGTAAACGTTGGGCACTAGGTGTGGGGGACATTCCACGTTTTCCGCGCCGTAGCTAACGCATTAAGTGCCCCGCCTGGGGAGTACGGCCGCAAGGCTAAAACTCAAAGGAATTGACGGGGGCCCGCACAAGCGGCGGAGCATGCGGATTAATTCGATGCAACGCGAAGAACCTTACCAAGGCTTGACATGTGCCAGACCGCTCCAGAGATGGGGTTTCCCTTCGGGGCTGGTTCACAGGTGGTGCATGGTTGTCGTCAGCTCGTGTCGTGAGATGTTGGGTTAAGTCCCGCAACGAGCGCAACCCTCGTTCCATGTTGCCAGCACGTAGTGGTGGGGACTCATGGGAGACTGCCGGGGTCAACTCGGAGGAAGGTGGGGATGACGTCAAATCATCATGCCCCTTATGTCTTGGGCTTCACGCATGCTACAATGGCCGGTACAATGGGTTGCGATACTGTGAGGTGGAGCTAATCCCTAAAAGCCGGTCTCAGTTCGGATTGGGGTCTGCAACTCGACCCCATGAAGTCGGAGTCGCTAGTAATCGCAGATCAGCAACGCTGCGGTGAATACGTTCCCGGGCCTTGTACACACCGCCCGTCAAGTCACGAAAGTTGGTAACACCCGAAGCCGATGGCCTAACCACCTTGTGTGGGGGGAGTCGTCGAAGG,null,aaca8aff99dbbf08ee2e17fd46648e14
22926897,URS00015DD631,2019-12-02T13:24:47.746Z,rnacen,0C89DA8C3FFEE6B7,460,CCTACGGGTGGCAGCAGTGAGGAATATTGGACAATGGGTGAAAGCCTGATCCAGCCATCCCGCGTGAAGGATGACGGTCCTATGGATTGTAAACTTCTTTTGTACAGGGATAAACCTACTCTCGTGAGGGTAGCTGAAGGTACTGTACGAATAAGCACCGGCTAACTCCGTGCCAGCAGCCGCGGTAATACGGAGGGTGCAAGCGTTATCCGGATTTATTGGGTTTAAAGGGTCCGTAGGCGGACCTGTGAGTCAGTGGTGAAATCTCATAGCTTAACTATGAAACTGCCATTGATACTGCAGGTCTTGAGTAAATTTGAAGTGGCTGGAATAAGTAGTGTAGCGGTGAAATGCATAGATATTACTTAGAACACCAATTGCGAAGGCAGGTCACTAAGATTTAACTGACGCTGATGGACGAAAGCGTGGGTAGCGAACAGGATTAGATACCCGGGTAGTC,null,aaca8b5602ccb5225faa288e10505074
22926898,URS00015DD632,2019-12-02T13:24:47.746Z,rnacen,AEC711F0958B94F3,253,TACGTAGGGGGCAAGCGTTGTCCGGAATCATTGGGCGTAAAGCGCGTGTAGGCGGTTCGGTAAGTCTGCTGTGAAAGTCCAGGGCTCAACCCTGGGATGCTGGTGGATACTGTCGGACTAGAGTACGGAAGAGGCGAGTGGAATTCCCGGTGTAGCGGTGAAATGCGCAGATATCAGGAGGAACACCGGTGGCGAAGGCGGGTCTCTGGGCCGATACTGACGCTGAGGAGCGAAAGCGTGGGGAGCGAACAGG,null,aaca8c75b7f0cf2cdc6df6295b992006
22926899,URS00015DD633,2019-12-02T13:24:47.746Z,rnacen,8A73A4A2398CF336,453,AGCGAACGCTGGCGGCAGGCCTAACACATGCAAGTCGAACGAAGTCTTCGGACTTATTGGCGGACGGGTGAGTAACACGTGGGAACGTACCTTTTGGTTCGGAACAACTAAGGGAAACTTGAGCTAATACCGGATGAGCCCCTAGGGGGAAAGATTTATCGCCGACAAGAGCGGCCCGCGTTAGATTAGCAAGTTGATGGGGTAAAGGCGCACCAAGGCTACGATCGATAGCTGGTCTGAGAGGAAGAACAGACACTCTGGAACTGAGACACGGCCGGGAGTCCTAGGGGAGGCAACCGTGCAGAATCTTGCGCCATGGGGGAAAGACTGACGCAGCCATGCCGCGTGAATGATGAAGGTCTTAGGATTGTAAAATTCTTTTACCAGGGACGATAATGACGGTACCTGAAGAAAAAGTCCCGGCTAACTTCGTGCCAGCAGCCGCCGTAAGAC,null,aaca8d7b74a68dd57e3333c6b0c062d8
22926900,URS00015DD634,2019-12-02T13:24:47.746Z,rnacen,98CD94407696E4A3,253,TACAGGGGGTGCTAGCGTTGTTCGGAATTACTGGGCGTAAAGGGCGCGTAGGTGGCTTGATGAGTCAGGAGTGAAATCCCGGAGCTTAACTCCGGAATTGCTTTTGAAACTATTAGGCTAGAGTATGTTAGAGGATGGCGGAATTCCTAGTGTAGAAGTGAAATTCGTAGATATTAGGAAGAACACCGGTGGCGAAGGCGGCCATCTGGGACATAACTGACACTGAGGCGCGAAAGCGTGGGGATCAAACAGG,null,aaca8de3e981625a031ce1f67c9ee896
22926901,URS00015DD635,2019-12-02T13:24:47.746Z,rnacen,F93F95768D526A9F,253,TACGAAGGGGGCTAGCGTTGTTCGGAATTACTGGGCGTAAAGGGCGCGCAGGCGGTCCTTCAAGTCAGGCGTGAAAGCCCCGGGCTCAACCTGGGAATCGCGCTTGAGACTGAGGGACTTGAGTTCGGGAGAGGAGAGCGGAATTCCCAGTGTAGAGGTGAAATGCGTAGATATCGGGAGGAACACCAGTGGCGAAGGCGACTACCTGGCCTGTTCTTGACGCTGAGGCGCGAAAGCTAGGGGAGCAAACGGG,null,aaca8eddadafb2df40d827c37cfdc7a4
22926902,URS00015DD636,2019-12-02T13:24:47.746Z,rnacen,BC63FEF70353EC1F,397,ATACGTAGGGGGCGAGCGTTGTCCGGAATGATTGGGCGTAAAGGGCGCGTAGGCGGCCCGGTAAGTCTGGAGTGAAAGTCCTGT